In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Análise Exploratória - Camada Bronze (Raw)
# MAGIC Explorando o JSON bruto salvo na tabela delta_brewery_bronze

# COMMAND
import json
from pyspark.sql.functions import col, lit

# 1. Carregar a tabela bronze
df_bronze = spark.table("delta_brewery_bronze")

# Visualizar as primeiras linhas da coluna raw_json crua
display(df_bronze.limit(5))

# COMMAND
# 2. Como a coluna é uma string JSON, podemos usar o parse dinâmico do Spark para extrair algumas chaves e analisar sem sujar a camada
# Vamos extrair o 'brewery_type' e o 'country' diretamente do JSON cru usando get_json_object:

from pyspark.sql.functions import get_json_object

df_extracted = df_bronze.select(
    get_json_object(col("raw_json"), "$.id").alias("id"),
    get_json_object(col("raw_json"), "$.name").alias("name"),
    get_json_object(col("raw_json"), "$.brewery_type").alias("brewery_type"),
    get_json_object(col("raw_json"), "$.country").alias("country"),
    get_json_object(col("raw_json"), "$.state_province").alias("state_province")
)

display(df_extracted.limit(10))

# COMMAND
# 3. Contagem por Tipo de Cervejaria direto do dado bruto
df_types = df_extracted.groupBy("brewery_type").count().orderBy(col("count").desc())
display(df_types)

# COMMAND
# 4. Contagem por País direto do dado bruto
df_countries = df_extracted.groupBy("country").count().orderBy(col("count").desc())
display(df_countries)